## Sentiment Analysis

### Classify documents based on their sentiment using ML

We will download the movie review dataset from IMDB.

In [ ]:
import tarfile
import pandas as pd
import numpy as np
import pyprind
import re
import os
import sys
import nltk
# import ssl

# try:
#     _create_unverified_https_context = ssl._create_unverified_context
# except AttributeError:
#     pass
# else:
#     ssl._create_default_https_context = _create_unverified_https_context

# # Now your download will bypass the SSL handshake error
# nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
# Unzip tar file 
with tarfile.open("data/aclImdb_v1.tar","r") as tar:
    tar.extractall(path="./data/")

In [10]:
basepath = 'data/aclImdb'

In [ ]:
## Reading reviews from both test and train dataset and appending them into one dataset and creating a label.
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream = sys.stdout)
df = pd.DataFrame()

for s in ('test', 'train'):
    for l in ('pos','neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding = 'utf-8') as infile:
                txt = infile.read()
            df = df.append([[txt, labels[l]]],
                           ignore_index = True)
            pbar.update()

df.columns = ['review', 'sentiment']

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:04:05


In [ ]:
## Reshuffle the index so we have a mix of both positive and negative review.
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
# save the data.
df.to_csv('data/movie_data.csv', index = False, encoding='utf-8')

In [23]:
df = pd.read_csv('data/movie_data.csv', encoding='utf-8')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


## Bag-of-words model

Bag of word model allow us to represent text as numerical features vectors.

The idea behind bag-of-words model is :-
1. we create a vocabulary of unique tokens - for example, words - from the entire set of documents.
2. we construct  a feature vector from each document that contains the count of how often each word occurs in the particular document.

Since the unique word in a document is a small subset of bag-of-words vocabulary, most feature vector will be 0, hence we call them sparse.

### Transform words into feature vectors

We can use CountVectorizer that take an array of text and create bag-of-words model for us.

In [ ]:
## An Example
count = CountVectorizer()
docs = np.array(['The sun is shining',
        'The weather is sweet',
        'The sun is shining, the weather is sweet, and one and one is two'])
bag = count.fit_transform(docs) # construct vocab of bag-of-world model and transformed them into feature vector

In [ ]:
# bag-of-words vocabulary
count.vocabulary_

{'the': 6,
 'sun': 4,
 'is': 1,
 'shining': 3,
 'weather': 8,
 'sweet': 5,
 'and': 0,
 'one': 2,
 'two': 7}

The values for each word is their index. which means that 'and' has index 0, 'is' has index 1 and so on. The indexes are usually assigned alphabetically. So feature vector the value in index 0 will represent how many occurance of word 'and' has in a particulat text.

In [ ]:
## Sparse feature vectors
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]


The values in the feature vectors are also called ***raw term frequencies***. 

***tf(t,d)*** - number of times a word t occur in the document d.

#### N-grams model

The contigous sequence of items in NLP - words, letters or symbols are also n-grams. The choice of number n depends on the kind of application. For e.g. in spam filtering, n-grams of size 3 or 4 yield good results. 

For a document "The sun is shining", 1-gram, 2-gram representation would be:-
* 1-gram: "The", "sun", "is", "shining".
* 2-gram: "The sun", "sun is", "is shining".


## term frequency-inverse document frequency (tf-idf)

During analysis, we often come across words that appear in all the documents from all classes. These frequently used words don't contain any valuable of differentiating information. We can use tf-idf to downweight such words in the feature vectors. The tf-idf can be defined as :-

$$tf-idf(t,d) = tf(t,d) * idf(t,d)$$
where idf(t,d) is given as 

$$idf(t,d) = log \frac{n_d}{1 + df(d,t)}$$
where $n_d$ is the total number of documents and df(t,d) is no. of documents contain the word t. The log is used to ensure low frequency words are not given too much weight. 1 is added just to make sure we have non-zero value of idf.


In [32]:
tfidf = TfidfTransformer(use_idf= True,
                         norm='l2',
                         smooth_idf=True)
np.set_printoptions(precision=2)
print(tfidf.fit_transform(count.fit_transform(docs)).toarray())


[[0.   0.43 0.   0.56 0.56 0.   0.43 0.   0.  ]
 [0.   0.43 0.   0.   0.   0.56 0.43 0.   0.56]
 [0.5  0.45 0.5  0.19 0.19 0.19 0.3  0.25 0.19]]


We can see that word "is" that has highest frequency in 3rd document is downweighted and now has tf-idf of 0.45 as it is also contained in other documents hence unlikely to have any useful information.

However, the scikit-learn has a different formula for idf.
$$idf(t,d) = log \frac{1 + n_d}{1 + df(d,t)}$$
and tf-idf would be 
$$tf-idf(t,d) = tf(t,d) * (idf(t,d) + 1)$$
The +1 is for setting smooth-idf = True, which is helpful for assigning 0 weight to terms occur in all documents.

It is also common to normalize term frequencies before calculating tf-idf. However, sklearn directly normalize tf-idfs. For L2 norm,
$$v_{norm} = \frac{v}{||v||_2}$$

## Cleaning text data

Text data may contain a lot of punctuation, HTML tags, other non letter characters. Some punctuations can be useful in some context.

In [34]:
df.loc[0,'review'][-50:]

'is seven.<br /><br />Title (Brazil): Not Available'

To remove these html, punctuation tags, we can use regex library. Word capitalization can be useful in some context but here we will convert everything to lower case.

In [39]:
def preprocessor(text):
    # For removing html/xml tags
    # map < bracket and Matches zero or more characters of any kind, except for a closing angle bracket using [^>]*
    # followed by a closing bracket >. which will remove html tags like <div>, etc.
    text = re.sub('<[^>]*>','',text) 

    # For removing emoticons by identifying eyes, nose, and mouth
    # (?::|;|=) - maps eyes as :, ;, =
    # (?:-) - maps nose as hyphen - exactly zero or one time because of the ? quantifier.
    # (?:\)|\(|D|P) - maps mouth as \, D, P
    # he ?: inside the parentheses creates non-capturing groups. 
    # This ensures re.findall returns the full matched emoticon string rather than just pieces of it.
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)

    # For removing all spaces, punctuation, symbols
    # [\W]+ - [] is for character class. \W means all non-word items.
    # + - Matches one or more of these non-word characters in a row to speed up substitution
    text = (re.sub('[\W]+', ' ', text.lower()) + 
            ' '.join(emoticons).replace('-', '')) # removing nose emoticons for consistency
    
    return text


In [ ]:
#Testing the function
preprocessor(df.loc[0,'review'][-50:])

'is seven title brazil not available'

In [41]:
#Cleaning the whole data
df['review'] = df['review'].apply(preprocessor)

### Processing documents into tokens

After the data preparation, we need to think about how to split text corpora into individual elements which is called ***tokenization***. 

* One way to tokenize is to split the text into individual terms.

In [42]:
def tokenizer(text):
    return(text.split())

tokenizer('runners like running and thus they run')

['runners', 'like', 'running', 'and', 'thus', 'they', 'run']

* Another technique of tokenization, is called word ***stemming***, which is a process of transforming a word to its root form. For e.g. 'running' becomes 'run'.

In [47]:
porter = PorterStemmer()
def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

tokenizer_porter('runners like running and thus they run')

['runner', 'like', 'run', 'and', 'thu', 'they', 'run']

### Stemming Algorithms

The porter stemming algorithm is the oldest one. There are newer and faster algorithm like Snowball stemmer, Lancaster stemmer. The new algorithm are more aggressive than porter algorithm as they produce shorter and obscure words.

### Lemmatization

While stemming create non-real words, lemmatization aim to obtain the canonical (grammatically correct) forms of individual words - which are called lemmas. However, lemmatization is computationally more difficult and expensive compared to stemming and also has little effect on performance fot text classification.

### Stop words

Stop words are the words that are extremly common and probably bear no useful information. Removing them can be useful if we are working with raw frequency rather than tf-idf, which already downweight the frequently occuring words.

In [ ]:
# removing stop words
stop = stopwords.words('english')
[w for w in tokenizer_porter('a runners like running and thus they run') if w not in stop]

['runner', 'like', 'run', 'thu', 'run']

### Training a logistic regression model for document classification

In [55]:
X_train = df.loc[:25000, 'review'].values
Y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
Y_test = df.loc[25000:, 'sentiment'].values

In [58]:
tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None)

small_param_grid = [
    {
        'vect__ngram_range': [(1,1)], # means 1-gram model, (1,2)  will consider both 1 and 2-gram model
        'vect__stop_words': [None],
        'vect__tokenizer': [tokenizer, tokenizer_porter],
        'clf__penalty': ['l2'],
        'clf__C': [1.0, 10.0]
    },
    {
        'vect__ngram_range': [(1,1)],
        'vect__stop_words': [stop, None],
        'vect__tokenizer': [tokenizer],
        'vect__use_idf': [False],
        'vect__norm': [None],
        'clf__penalty': ['l2'],
        'clf__C': [1.0, 10.0]
    }
]

lr_tfidf = Pipeline([
    ('vect', tfidf),
    ('clf', LogisticRegression(solver='liblinear'))
])

gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid,
                           scoring='accuracy', cv=5,
                           verbose=2, n_jobs = 1)
gs_lr_tfidf.fit(X_train, Y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   7.2s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   6.4s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   5.0s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   4.9s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   5.1s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer_porter 

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vect',
                                        TfidfVectorizer(lowercase=False)),
                                       ('clf',
                                        LogisticRegression(solver='liblinear'))]),
             n_jobs=1,
             param_grid=[{'clf__C': [1.0, 10.0], 'clf__penalty': ['l2'],
                          'vect__ngram_range': [(1, 1)],
                          'vect__stop_words': [None],
                          'vect__tokenizer': [<function tokenizer at 0x7faabcfc16a8>,
                                              <function tokenizer_porter at 0x7faac8067d08>...
                          'vect__stop_words': [['a', 'about', 'above', 'after',
                                                'again', 'against', 'ain',
                                                'all', 'am', 'an', 'and', 'any',
                                                'are', 'aren', "aren't", 'as',
                         

The idea of the grid above is to test if stemming, idf has an effect or not over raw frequency and no stemming.

In [59]:
print(gs_lr_tfidf.best_params_)

{'clf__C': 10.0, 'clf__penalty': 'l2', 'vect__ngram_range': (1, 1), 'vect__stop_words': None, 'vect__tokenizer': <function tokenizer at 0x7faabcfc16a8>}


We got the best results with tf-idf with no stop words, and stemming combined with LogisticRegression that use L2 regularization with C as 10.

In [60]:
print(f'CV accuracy is : {gs_lr_tfidf.best_score_: .3f}')

CV accuracy is :  0.897


In [61]:
clf = gs_lr_tfidf.best_estimator_
print(f'Test accuracy : {clf.score(X_test, Y_test): .3f}')

Test accuracy :  0.899
